# Evaluate neutrino direction model

Reads the results CSV and metrics JSON produced by `train_neutrino_direction.py`.
No graphnet/icecube dependencies needed — just pandas, numpy, matplotlib.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("/groups/icecube/holgerkc/Thesis_Analysis/Classifiers/finding_the_angles")
PREFIX = "neutrino_direction"

results = pd.read_csv(OUTPUT_DIR / f"{PREFIX}_results.csv")
metrics = json.loads((OUTPUT_DIR / f"{PREFIX}_metrics.json").read_text())
train_config = json.loads((OUTPUT_DIR / f"{PREFIX}_train_config.json").read_text())

print(f"Test events: {len(results)}")
print(f"Training: epochs={train_config['epochs']}, max_events={train_config['max_events']}, batch_size={train_config['batch_size']}")
print(f"Features: {train_config['features']}")

---
## Metrics summary

In [ ]:
print(json.dumps(metrics, indent=2))

---
## Error distributions

In [ ]:
az_err = results["azimuth_abs_err_deg"].to_numpy()
ze_err = results["zenith_abs_err_deg"].to_numpy()
op_err = results["opening_angle_err_deg"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Azimuth error
axes[0, 0].hist(az_err, bins=50, color="tab:blue", alpha=0.8)
axes[0, 0].axvline(np.mean(az_err), color="black", ls="--", label=f"Mean = {np.mean(az_err):.2f}\u00b0")
axes[0, 0].set_title("Azimuth absolute error")
axes[0, 0].set_xlabel("Error [deg]")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend()

# Zenith error
axes[0, 1].hist(ze_err, bins=50, color="tab:orange", alpha=0.8)
axes[0, 1].axvline(np.mean(ze_err), color="black", ls="--", label=f"Mean = {np.mean(ze_err):.2f}\u00b0")
axes[0, 1].set_title("Zenith absolute error")
axes[0, 1].set_xlabel("Error [deg]")
axes[0, 1].set_ylabel("Count")
axes[0, 1].legend()

# Opening angle error
axes[1, 0].hist(op_err, bins=50, color="tab:green", alpha=0.8)
axes[1, 0].axvline(np.mean(op_err), color="black", ls="--", label=f"Mean = {np.mean(op_err):.2f}\u00b0")
axes[1, 0].axvline(np.quantile(op_err, 0.68), color="tab:red", ls=":", label=f"68% = {np.quantile(op_err, 0.68):.2f}\u00b0")
axes[1, 0].set_title("Opening-angle error")
axes[1, 0].set_xlabel("Error [deg]")
axes[1, 0].set_ylabel("Count")
axes[1, 0].legend()

# CDF
x_cdf = np.sort(op_err)
y_cdf = np.arange(1, len(x_cdf) + 1) / len(x_cdf)
axes[1, 1].plot(x_cdf, y_cdf, color="tab:purple")
axes[1, 1].set_title("Opening-angle error CDF")
axes[1, 1].set_xlabel("Error [deg]")
axes[1, 1].set_ylabel("Cumulative fraction")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Opening angle vs energy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    results["energy"], op_err,
    c=results["pid"], cmap="tab10", alpha=0.5, s=10,
)
ax.set_xlabel("True energy [log10(GeV)]")
ax.set_ylabel("Opening-angle error [deg]")
ax.set_title("Direction error vs energy")
plt.colorbar(sc, label="pid")
plt.tight_layout()
plt.show()

---
## Per-flavour performance

In [ ]:
pid_names = {12: "nu_e", -12: "nu_e_bar", 14: "nu_mu", -14: "nu_mu_bar", 16: "nu_tau", -16: "nu_tau_bar"}

for pid_val in sorted(results["pid"].unique()):
    mask = results["pid"] == pid_val
    n = mask.sum()
    if n == 0:
        continue
    med = np.median(results.loc[mask, "opening_angle_err_deg"])
    q68 = np.quantile(results.loc[mask, "opening_angle_err_deg"], 0.68)
    name = pid_names.get(int(pid_val), str(int(pid_val)))
    print(f"{name:>12s} (pid={int(pid_val):>3d}): n={n:>5d}, median={med:.2f} deg, q68={q68:.2f} deg")